In [1]:
import sys
sys.path.append('../01-agentic-rag')


In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

161

In [5]:
documents = documents_llm

In [6]:
doc = documents[5]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

69d122f12e
Certificate: Can I follow the course in a self-paced mode and get a certificate?
No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to finish a capstone project and complete the
required peer reviews. Homework is not required. You can work through the
material and prepare your project in self-paced mode, but project submission and
peer review must happen while a live cohort is accepting them.


In [7]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
import json
user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
response.output_parsed.questions

['I just found this course late — is it still possible to join now?',
 'Can I enroll after the course has already started?',
 'If I join late, can I still get a certificate somehow?',
 'What do I need to do to be eligible for the certificate if I’m starting now?',
 'Is there still time to submit the project for certification if I’ve only discovered the course recently?']

In [16]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [17]:
from evaluation_utils import llm_structured

In [18]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — am I still able to join, or is it too late?', 'If I sign up now, can I still take part in the course activities?', 'Is late enrollment allowed for this course, or only if I joined from the start?', 'Can new students still join the course after it has already started?', 'If I start the course now, will I still be able to get a certificate?']


In [19]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=98, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=305)

In [20]:
from evaluation_utils import calc_price

In [21]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.000441, 'total_cost': 0.00059625}

In [22]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — am I still able to join, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'If I sign up now, can I still take part in the course activities?',
  'document': '74eb249bbf'},
 {'question': 'Is late enrollment allowed for this course, or only if I joined from the start?',
  'document': '74eb249bbf'},
 {'question': 'Can new students still join the course after it has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course now, will I still be able to get a certificate?',
  'document': '74eb249bbf'}]

In [23]:
from evaluation_utils import llm_structured_retry

In [24]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [25]:
generate_ground_truth(doc)

([{'question': 'Can I join the course late if I just found it now?',
   'document': '74eb249bbf'},
  {'question': 'Is it still okay to enroll after the course has already started?',
   'document': '74eb249bbf'},
  {'question': 'If I start the course now, can I still get a certificate somehow?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to be eligible for the certificate if I join late?',
   'document': '74eb249bbf'},
  {'question': 'Does late entry affect the project submission deadline for the certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=86, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=293))

In [26]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [27]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [28]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/161 [00:00<?, ?it/s]

In [29]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

805

In [30]:
ground_truth[10]

{'question': "Where do students watch the Office Hours live if they don't have the Zoom link?",
 'document': '489dd1c9d9'}

In [31]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.12687150000000003

In [32]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.12687150000000003

In [33]:
import pandas as pd

In [34]:
df_ground_truth = pd.DataFrame(ground_truth)

In [39]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [40]:
len(df_ground_truth)

805